# Part 3 — Benchmark Forecasting Models
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/03_benchmark_models.ipynb)

Five benchmark forecasts — mean, naive, daily seasonal naive, weekly seasonal naive, drift — evaluated on the held-out 14-day test period. These set the bar every later model (SARIMAX, feature-based, foundation) has to clear. Self-contained: rebuilds the hourly dataset if it isn't already present.

**Evaluation protocol:** all forecasts are a single continuous 336-hour-ahead forecast from one fixed origin (end of training) — matching the supplementary demo pipeline — rather than a rolling 24-hour re-forecast. That's a harder test than the brief's '24 hour horizon' framing implies on its own.

In [ ]:
!pip install -q statsmodels

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_CSV_URL = 'https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv'

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
for d in [DATA_DIR, OUTPUT_DIR / 'forecasts', OUTPUT_DIR / 'metrics', OUTPUT_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## Load hourly data (self-contained — see Part 1/2 for the resampling logic)

In [ ]:
hourly_path = DATA_DIR / 'energydata_hourly.csv'

if hourly_path.exists():
    hourly = pd.read_csv(hourly_path, index_col=0, parse_dates=True)
    print('Loaded existing hourly dataset from', hourly_path)
else:
    print('No local hourly dataset found — rebuilding from the raw source...')
    raw = pd.read_csv(RAW_CSV_URL)
    raw['date'] = pd.to_datetime(raw['date'])
    raw = raw.set_index('date').sort_index()
    energy_cols = ['Appliances', 'lights']
    sensor_cols = [c for c in raw.columns if c not in energy_cols + ['rv1', 'rv2']]
    hourly = pd.concat([
        raw[energy_cols].resample('h').sum(),
        raw[sensor_cols].resample('h').mean(),
    ], axis=1)
    hourly.to_csv(hourly_path)

print(f'Hourly shape: {hourly.shape}')

## Problem constants, split, and metrics (same definitions as Part 2)

In [ ]:
TARGET = 'Appliances'
DAILY_PERIOD = 24
WEEKLY_PERIOD = 168
TEST_DAYS = 14

def train_test_split_by_days(series, test_days=TEST_DAYS):
    test_steps = test_days * DAILY_PERIOD
    return series.iloc[:-test_steps], series.iloc[-test_steps:]

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true.values - y_pred.values)))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true.values - y_pred.values) ** 2)))

def mase(y_true, y_pred, y_train, seasonality=DAILY_PERIOD):
    y_train = y_train.astype(float)
    naive_errors = np.abs(y_train.iloc[seasonality:].values - y_train.iloc[:-seasonality].values)
    scale = naive_errors.mean()
    return float('nan') if scale == 0 else float(np.mean(np.abs(y_true.values - y_pred.values)) / scale)

def bias(y_true, y_pred):
    return float(np.mean(y_pred.values - y_true.values))

def evaluate_forecast(name, y_true, y_pred, y_train):
    y_pred = y_pred.reindex(y_true.index)
    valid = y_true.notna() & y_pred.notna()
    y_true_v, y_pred_v = y_true.loc[valid], y_pred.loc[valid]
    return {
        'model': name,
        'MAE': mae(y_true_v, y_pred_v), 'RMSE': rmse(y_true_v, y_pred_v),
        'MASE': mase(y_true_v, y_pred_v, y_train, seasonality=DAILY_PERIOD),
        'Bias': bias(y_true_v, y_pred_v), 'n_points': int(valid.sum()),
    }

train, test = train_test_split_by_days(hourly[TARGET], TEST_DAYS)
horizon = len(test)
print(f'Train: {train.index.min()} to {train.index.max()} ({len(train)} obs)')
print(f'Test:  {test.index.min()} to {test.index.max()} ({len(test)} obs, horizon={horizon})')

## Benchmark forecasts

In [ ]:
def mean_forecast(y_train, horizon, index):
    """Forecast every step as the training mean. The floor any model must clear."""
    return pd.Series(y_train.mean(), index=index, name='mean')

def naive_forecast(y_train, horizon, index):
    """Forecast every step as the last observed training value."""
    return pd.Series(y_train.iloc[-1], index=index, name='naive')

def seasonal_naive_forecast(y_train, horizon, index, seasonality):
    """Recursive seasonal-naive: step i is the value `seasonality` steps
    before it. Beyond one season, this tiles the last complete season's
    actual values across the whole horizon."""
    history = list(y_train.values)
    values = []
    for _ in range(horizon):
        values.append(history[-seasonality])
        history.append(values[-1])
    return pd.Series(values, index=index)

def drift_forecast(y_train, horizon, index):
    """Extrapolate the straight line between the first and last training points."""
    slope = (y_train.iloc[-1] - y_train.iloc[0]) / (len(y_train) - 1)
    values = [y_train.iloc[-1] + slope * step for step in range(1, horizon + 1)]
    return pd.Series(values, index=index, name='drift')

forecasts = {
    'mean': mean_forecast(train, horizon, test.index),
    'naive': naive_forecast(train, horizon, test.index),
    'seasonal_naive_daily': seasonal_naive_forecast(train, horizon, test.index, DAILY_PERIOD),
    'seasonal_naive_weekly': seasonal_naive_forecast(train, horizon, test.index, WEEKLY_PERIOD),
    'drift': drift_forecast(train, horizon, test.index),
}

## Evaluate and compare

In [ ]:
results = [evaluate_forecast(name, test, fc, train) for name, fc in forecasts.items()]
results_df = pd.DataFrame(results).sort_values('MASE').reset_index(drop=True)
print('Benchmark comparison (sorted by MASE, lower is better):')
results_df.round(3)

In [ ]:
strongest = results_df.iloc[0]
print(f"Strongest benchmark: {strongest['model']}  (MASE={strongest['MASE']:.3f}, MAE={strongest['MAE']:.1f} Wh)")
print('This is the bar every later model (SARIMAX, feature-based, foundation) must clear.')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
train.tail(7 * DAILY_PERIOD).plot(ax=ax, label='Train (last 7 days)', color='#999999', linewidth=1)
test.plot(ax=ax, label='Actual (test)', color='black', linewidth=1.6)
colors = ['#1f5b8a', '#c0392b', '#2e8a5b', '#8a5b1f', '#8a1f6e']
for (name, fc), color in zip(forecasts.items(), colors):
    fc.plot(ax=ax, label=name, alpha=0.85, linewidth=1, color=color)
ax.set_title('Benchmark forecasts vs. actual — 336h continuous test-period forecast')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
ax.legend(loc='upper right', fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '03_benchmark_forecast_comparison.png')
plt.show()

## Save outputs

In [ ]:
forecast_df = pd.DataFrame({'actual': test})
for name, fc in forecasts.items():
    forecast_df[name] = fc.reindex(test.index)
forecast_df.to_csv(OUTPUT_DIR / 'forecasts' / 'benchmark_forecasts.csv')
results_df.to_csv(OUTPUT_DIR / 'metrics' / 'benchmark_metrics.csv', index=False)
print('Saved forecasts and metrics to outputs/')